# Report Design, Interactivity, Security, Power BI Service, Governance and Other BI Tools

Notebook for **Chapter 11** of the *analyst* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/analyst/ch11.html

Tip: change a number, predict the output, then run the cell again.

## 11.1 Report Design: Layout, Visuals, Colour and Accessibility

## 11.2 Interactivity: Slicers, Drill, Bookmarks, Tooltips and Parameters

## 11.3 Row-Level Security and the Power BI Service

## 11.4 Governance, Documentation and Performance

## 11.5 Tableau, Looker Studio and Other BI Tools

## 11.6 Start Here: The Plain-English Map and Glossary

In [ ]:
import pandas as pd
orders = pd.DataFrame({
    "OrderID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    "Date": pd.to_datetime(["2025-01-03", "2025-01-03", "2025-01-04", "2025-01-05",
                            "2025-01-06", "2025-01-07", "2025-01-09", "2025-01-10"]),
    "City": ["Pune", "Mumbai", "Pune", "Delhi", "Mumbai", "Mumbai", "Delhi", "Pune"],
    "Status": ["Delivered", "Delivered", "Returned", "Delivered", "Cancelled", "Delivered", "Delivered", "Delivered"],
    "Qty": [2, 1, 3, 1, 5, 2, 1, 4],
    "Amount": [998, 1299, 597, 799, 450, 1298, 2499, 1400]})
region = {"Delhi": "North", "Mumbai": "West", "Pune": "West"}
orders["Region"] = orders["City"].map(region)

print(orders.groupby("City").Amount.agg(["count", "sum"]).to_string())
print(orders.groupby("Region").Amount.agg(["count", "sum"]).to_string())
print(orders.groupby("Status").Amount.agg(["count", "sum"]).to_string())
print("total", orders.Amount.sum(), "qty", orders.Qty.sum())

        count   sum
City               
Delhi       2  3298
Mumbai      3  3047
Pune        3  2995
        count   sum
Region             
North       2  3298
West        6  6042
           count   sum
Status                
Cancelled      1   450
Delivered      6  8293
Returned       1   597
total 9340 qty 19


## 11.7 Report Design Mastery: Grids, Contrast, Visual Choice and Themes

In [ ]:
def grid(total, margin, gap, n):
    width = (total - 2 * margin - (n - 1) * gap) / n
    starts = [margin + k * (width + gap) for k in range(n)]
    return width, starts

print(grid(1280, 20, 20, 3))
print(grid(1280, 20, 20, 4))
w, s = grid(1280, 15, 15, 3)
print(round(w, 1), [round(x, 1) for x in s])

(400.0, [20.0, 440.0, 860.0])
(295.0, [20.0, 335.0, 650.0, 965.0])
406.7 [15.0, 436.7, 858.3]


In [ ]:
def lin(c):
    c = c / 255
    return c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4

def lum(hexcode):
    h = hexcode.lstrip("#")
    r, g, b = (lin(int(h[i:i + 2], 16)) for i in (0, 2, 4))
    return 0.2126 * r + 0.7152 * g + 0.0722 * b

def contrast(a, b):
    hi, lo = sorted([lum(a), lum(b)], reverse=True)
    return (hi + 0.05) / (lo + 0.05)

for name, col in [("light grey", "#9aa5b1"), ("mid grey", "#767676"), ("dark grey", "#595959"),
                  ("blue", "#1f6feb"), ("house blue", "#2b4fd6"), ("house orange", "#b84a22"),
                  ("house teal", "#357a70")]:
    print(f"{name:12s} {col}  on white {contrast(col, '#ffffff'):5.2f}")
print("white text on house blue", round(contrast("#ffffff", "#2b4fd6"), 2))
print("dark grey on light panel F2F2F2", round(contrast("#595959", "#f2f2f2"), 2))
print("light grey on light panel F2F2F2", round(contrast("#9aa5b1", "#f2f2f2"), 2))

light grey   #9aa5b1  on white  2.50
mid grey     #767676  on white  4.54
dark grey    #595959  on white  7.00
blue         #1f6feb  on white  4.63
house blue   #2b4fd6  on white  6.58
house orange #b84a22  on white  5.20
house teal   #357a70  on white  5.04
white text on house blue 6.58
dark grey on light panel F2F2F2 6.26
light grey on light panel F2F2F2 2.24


In [ ]:
import json

def lin(c):
    c = c / 255
    return c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4

def lum(h):
    h = h.lstrip("#")
    r, g, b = (lin(int(h[i:i + 2], 16)) for i in (0, 2, 4))
    return 0.2126 * r + 0.7152 * g + 0.0722 * b

theme = {"name": "Shop house style",
         "dataColors": ["#2b4fd6", "#b84a22", "#357a70", "#aa2d26", "#6a2fd0"],
         "background": "#ffffff", "foreground": "#595959", "tableAccent": "#2b4fd6"}
for c in theme["dataColors"]:
    print(c, round(1.05 / (lum(c) + 0.05), 2), "ok" if 1.05 / (lum(c) + 0.05) >= 3 else "WEAK")
print(json.dumps({"name": theme["name"], "n_colours": len(theme["dataColors"])}))

#2b4fd6 6.58 ok
#b84a22 5.2 ok
#357a70 5.04 ok
#aa2d26 6.75 ok
#6a2fd0 7.17 ok
{"name": "Shop house style", "n_colours": 5}


## 11.8 Interactivity Mastery: Filters, Interactions, Drill and Parameters

In [ ]:
import pandas as pd
orders = pd.DataFrame({
    "OrderID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    "Date": pd.to_datetime(["2025-01-03", "2025-01-03", "2025-01-04", "2025-01-05",
                            "2025-01-06", "2025-01-07", "2025-01-09", "2025-01-10"]),
    "City": ["Pune", "Mumbai", "Pune", "Delhi", "Mumbai", "Mumbai", "Delhi", "Pune"],
    "Status": ["Delivered", "Delivered", "Returned", "Delivered", "Cancelled", "Delivered", "Delivered", "Delivered"],
    "Qty": [2, 1, 3, 1, 5, 2, 1, 4],
    "Amount": [998, 1299, 597, 799, 450, 1298, 2499, 1400]})
region = {"Delhi": "North", "Mumbai": "West", "Pune": "West"}
orders["Region"] = orders["City"].map(region)

def visual(df, **filters):
    # each keyword is column=allowed values; rows must satisfy every filter (AND)
    for col, allowed in filters.items():
        df = df[df[col].isin(allowed)]
    return df

print("no filters      ", visual(orders).Amount.sum())
print("slicer West     ", visual(orders, Region=["West"]).Amount.sum())
print("+ page Delivered", visual(orders, Region=["West"], Status=["Delivered"]).Amount.sum())
print("+ cross-filter Pune", visual(orders, Region=["West"], Status=["Delivered"], City=["Pune"]).Amount.sum())

no filters       9340
slicer West      6042
+ page Delivered 4995
+ cross-filter Pune 2398


In [ ]:
import pandas as pd
orders = pd.DataFrame({
    "OrderID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    "Date": pd.to_datetime(["2025-01-03", "2025-01-03", "2025-01-04", "2025-01-05",
                            "2025-01-06", "2025-01-07", "2025-01-09", "2025-01-10"]),
    "City": ["Pune", "Mumbai", "Pune", "Delhi", "Mumbai", "Mumbai", "Delhi", "Pune"],
    "Status": ["Delivered", "Delivered", "Returned", "Delivered", "Cancelled", "Delivered", "Delivered", "Delivered"],
    "Qty": [2, 1, 3, 1, 5, 2, 1, 4],
    "Amount": [998, 1299, 597, 799, 450, 1298, 2499, 1400]})
region = {"Delhi": "North", "Mumbai": "West", "Pune": "West"}
orders["Region"] = orders["City"].map(region)

daily = orders.groupby(orders.Date.dt.day).Amount.sum()
pune = orders[orders.City == "Pune"].groupby(orders.Date.dt.day).Amount.sum()
print("None      :", daily.to_dict())
print("Highlight :", {d: (int(pune.get(d, 0)), int(v)) for d, v in daily.items()})
print("Filter    :", pune.to_dict())
print("card (Filter):", orders[orders.City == "Pune"].Amount.sum(), " card (None):", orders.Amount.sum())

None      : {3: 2297, 4: 597, 5: 799, 6: 450, 7: 1298, 9: 2499, 10: 1400}
Highlight : {3: (998, 2297), 4: (597, 597), 5: (0, 799), 6: (0, 450), 7: (0, 1298), 9: (0, 2499), 10: (1400, 1400)}
Filter    : {3: 998, 4: 597, 10: 1400}
card (Filter): 2995  card (None): 9340


In [ ]:
import pandas as pd
orders = pd.DataFrame({
    "OrderID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    "Date": pd.to_datetime(["2025-01-03", "2025-01-03", "2025-01-04", "2025-01-05",
                            "2025-01-06", "2025-01-07", "2025-01-09", "2025-01-10"]),
    "City": ["Pune", "Mumbai", "Pune", "Delhi", "Mumbai", "Mumbai", "Delhi", "Pune"],
    "Status": ["Delivered", "Delivered", "Returned", "Delivered", "Cancelled", "Delivered", "Delivered", "Delivered"],
    "Qty": [2, 1, 3, 1, 5, 2, 1, 4],
    "Amount": [998, 1299, 597, 799, 450, 1298, 2499, 1400]})
region = {"Delhi": "North", "Mumbai": "West", "Pune": "West"}
orders["Region"] = orders["City"].map(region)

measures = {"Sales": lambda d: d.Amount.sum(),
            "Orders": lambda d: d.OrderID.nunique(),
            "Quantity": lambda d: d.Qty.sum()}
for pick, fn in measures.items():
    print(pick.ljust(9), {c: int(fn(g)) for c, g in orders.groupby("City")}, "total", int(fn(orders)))

Sales     {'Delhi': 3298, 'Mumbai': 3047, 'Pune': 2995} total 9340
Orders    {'Delhi': 2, 'Mumbai': 3, 'Pune': 3} total 8
Quantity  {'Delhi': 2, 'Mumbai': 8, 'Pune': 9} total 19


## 11.9 Row-Level Security Mastery: Design, Test, Break and Fix

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript('''
CREATE TABLE dim_city(city TEXT, region TEXT);
INSERT INTO dim_city VALUES ('Delhi','North'),('Mumbai','West'),('Pune','West');
CREATE TABLE fact(order_id INT, city TEXT, amount INT);
INSERT INTO fact VALUES (1001,'Pune',998),(1002,'Mumbai',1299),(1003,'Pune',597),(1004,'Delhi',799),
                        (1005,'Mumbai',450),(1006,'Mumbai',1298),(1007,'Delhi',2499),(1008,'Pune',1400);
CREATE TABLE security(email TEXT, region TEXT);
INSERT INTO security VALUES ('asha@corp.example','West'),('ravi@corp.example','North'),
                            ('cfo@corp.example','North'),('cfo@corp.example','West');
''')
sql = '''
SELECT COUNT(*) AS n_rows, COALESCE(SUM(f.amount), 0) AS total
FROM fact f JOIN dim_city d ON d.city = f.city
WHERE d.region IN (SELECT region FROM security WHERE email = ?)
'''
for user in ["asha@corp.example", "ravi@corp.example", "cfo@corp.example", "new.joiner@corp.example"]:
    n, t = con.execute(sql, (user,)).fetchone()
    print(f"{user:25s} rows={n} total={t}")

asha@corp.example         rows=6 total=6042
ravi@corp.example         rows=2 total=3298
cfo@corp.example          rows=8 total=9340
new.joiner@corp.example   rows=0 total=0


In [ ]:
import pandas as pd
emp = pd.DataFrame({"EmpID": [1, 2, 3, 4, 5, 6],
                    "Name": ["Meera", "Ravi", "Asha", "Kiran", "Lata", "Omar"],
                    "ManagerID": [None, 1, 1, 2, 2, 4]})
parent = dict(zip(emp.EmpID, emp.ManagerID))
name = dict(zip(emp.EmpID, emp.Name))

def path(e):
    chain = [e]
    while parent[chain[-1]] == parent[chain[-1]]:   # NaN != NaN stops at the top
        chain.append(int(parent[chain[-1]]))
    return "|".join(str(x) for x in reversed(chain))

owner = {1001: 6, 1002: 3, 1003: 4, 1004: 5, 1005: 3, 1006: 6, 1007: 5, 1008: 4}
amount = {1001: 998, 1002: 1299, 1003: 597, 1004: 799, 1005: 450, 1006: 1298, 1007: 2499, 1008: 1400}
print({name[e]: path(e) for e in emp.EmpID})
for viewer in [1, 2, 3, 6]:
    seen = [o for o, rep in owner.items() if str(viewer) in path(rep).split("|")]
    print(name[viewer].ljust(6), "sees orders", seen, "total", sum(amount[o] for o in seen))

{'Meera': '1', 'Ravi': '1|2', 'Asha': '1|3', 'Kiran': '1|2|4', 'Lata': '1|2|5', 'Omar': '1|2|4|6'}
Meera  sees orders [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008] total 9340
Ravi   sees orders [1001, 1003, 1004, 1006, 1007, 1008] total 7591
Asha   sees orders [1002, 1005] total 1749
Omar   sees orders [1001, 1006] total 2296


In [ ]:
import pandas as pd
orders = pd.DataFrame({"City": ["Pune", "Mumbai", "Pune", "Delhi", "Mumbai", "Mumbai", "Delhi", "Pune"],
                       "Amount": [998, 1299, 597, 799, 450, 1298, 2499, 1400]})
orders["Region"] = orders.City.map({"Delhi": "North", "Mumbai": "West", "Pune": "West"})
company_total = orders.Amount.sum()                       # kept in a separate, unsecured summary table
for user, regions in [("asha", ["West"]), ("ravi", ["North"]), ("cfo", ["North", "West"])]:
    visible = orders[orders.Region.isin(regions)]
    west = visible[visible.Region == "West"].Amount.sum()
    naive = west / visible.Amount.sum()                    # ALL() cannot see hidden rows
    right = west / company_total
    print(f"{user:5s} West share naive={naive:6.1%}  from unsecured total={right:6.1%}" )

asha  West share naive=100.0%  from unsecured total= 64.7%
ravi  West share naive=  0.0%  from unsecured total=  0.0%
cfo   West share naive= 64.7%  from unsecured total= 64.7%


## 11.10 The Power BI Service: Workspaces, Refresh, Storage Modes and Sharing

In [ ]:
from datetime import datetime, timedelta

def plan(landing, steps, deadline, buffer_min=15):
    t = datetime.strptime(landing, "%H:%M") + timedelta(minutes=buffer_min)
    start = t
    for name, minutes in steps:
        t += timedelta(minutes=minutes)
        print(f"  {name:9s} ends {t:%H:%M}")
    slack = (datetime.strptime(deadline, "%H:%M") - t).total_seconds() / 60
    print(f"  start {start:%H:%M}, slack {slack:.0f} min")
    return slack

print("Case 1: dataset only")
plan("06:00", [("dataset", 25)], "07:00")
print("Case 2: dataflow then dataset")
plan("06:00", [("dataflow", 15), ("dataset", 25)], "07:00")
print("Case 3: slow dataset (45 min)")
plan("06:00", [("dataset", 45)], "07:00")
print("Case 4: after incremental refresh (10 min)")
plan("06:00", [("dataset", 10)], "07:00")

Case 1: dataset only
  dataset   ends 06:40
  start 06:15, slack 20 min
Case 2: dataflow then dataset
  dataflow  ends 06:30
  dataset   ends 06:55
  start 06:15, slack 5 min
Case 3: slow dataset (45 min)
  dataset   ends 07:00
  start 06:15, slack 0 min
Case 4: after incremental refresh (10 min)
  dataset   ends 06:25
  start 06:15, slack 35 min


## 11.11 Governance and Performance Mastery: Definitions, Dictionaries and Triage

In [ ]:
import pandas as pd
orders = pd.DataFrame({"Status": ["Delivered", "Delivered", "Returned", "Delivered", "Cancelled", "Delivered", "Delivered", "Delivered"],
                       "Amount": [998, 1299, 597, 799, 450, 1298, 2499, 1400]})
total = orders.Amount.sum()
delivered = orders[orders.Status == "Delivered"].Amount.sum()
cancelled = orders[orders.Status == "Cancelled"].Amount.sum()
returned = orders[orders.Status == "Returned"].Amount.sum()
defs = {
    "A value delivered / all value": delivered / total,
    "B value delivered / value excl. cancelled": delivered / (total - cancelled),
    "C orders delivered / all orders": (orders.Status == "Delivered").mean(),
    "D value delivered / value excl. cancelled+returned": delivered / (total - cancelled - returned),
}
for k, v in defs.items():
    print(f"{k:52s} {v:6.1%}")
print("sales views:", total, total - cancelled, total - cancelled - returned)

A value delivered / all value                         88.8%
B value delivered / value excl. cancelled             93.3%
C orders delivered / all orders                       75.0%
D value delivered / value excl. cancelled+returned   100.0%
sales views: 9340 8890 8293


In [ ]:
import pandas as pd
d = pd.DataFrame([
    ("Total Sales", "Sum of Amount, all statuses", "SUM(FactSales[Amount])", "Sales analytics"),
    ("Delivered Rate", "Delivered value over total", "DIVIDE(CALCULATE([Total Sales], FactSales[Status]=\"Delivered\"), [Total Sales])", ""),
    ("Total Sales", "Sales after returns", "", "Finance"),
    ("Orders", "Count of orders", "DISTINCTCOUNT(FactSales[OrderID])", "Sales analytics"),
], columns=["Name", "Definition", "Formula", "Owner"])
print("missing owner  :", d.loc[d.Owner == "", "Name"].tolist())
print("missing formula:", d.loc[d.Formula == "", "Name"].tolist())
print("duplicate names:", d.Name[d.Name.duplicated(keep=False)].unique().tolist())

missing owner  : ['Delivered Rate']
missing formula: ['Total Sales']
duplicate names: ['Total Sales']


In [ ]:
def page_time(times, fixed=None):
    fixed = fixed or {}
    return sum(fixed.get(k, v) for k, v in times.items())

times = {"matrix": 9.5, "s1": 0.4, "s2": 0.4, "s3": 0.4, "s4": 0.4, "s5": 0.4}
base = page_time(times)
print("before                :", base)
print("fix matrix to 1.5     :", page_time(times, {"matrix": 1.5}))
print("fix all small to 0.1  :", page_time(times, {f"s{i}": 0.1 for i in range(1, 6)}))
print("matrix share of page  :", round(times["matrix"] / base, 3))
print("best possible from small fixes:", round(base - page_time(times, {f"s{i}": 0 for i in range(1, 6)}), 1))

before                : 11.5
fix matrix to 1.5     : 3.5
fix all small to 0.1  : 10.0
matrix share of page  : 0.826
best possible from small fixes: 2.0


## 11.12 BI Tools Compared in Depth: LOD Expressions, Blending and Choosing

In [ ]:
import pandas as pd
o = pd.DataFrame({"OrderID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
                  "City": ["Pune", "Mumbai", "Pune", "Delhi", "Mumbai", "Mumbai", "Delhi", "Pune"],
                  "Status": ["Delivered", "Delivered", "Returned", "Delivered", "Cancelled", "Delivered", "Delivered", "Delivered"],
                  "Amount": [998, 1299, 597, 799, 450, 1298, 2499, 1400]})
# FIXED city: city total attached to every row
o["city_total"] = o.groupby("City").Amount.transform("sum")
o["share_of_city"] = (o.Amount / o.city_total).round(3)
# EXCLUDE city: total of the view without City on it (grand total here)
o["share_of_grand"] = (o.Amount / o.Amount.sum()).round(3)
print(o[["OrderID", "City", "Amount", "city_total", "share_of_city", "share_of_grand"]].to_string(index=False))
# INCLUDE status: per city, average of the status-level totals
inc = o.groupby(["City", "Status"]).Amount.sum().groupby("City").mean()
print(inc.to_dict())

 OrderID   City  Amount  city_total  share_of_city  share_of_grand
    1001   Pune     998        2995          0.333           0.107
    1002 Mumbai    1299        3047          0.426           0.139
    1003   Pune     597        2995          0.199           0.064
    1004  Delhi     799        3298          0.242           0.086
    1005 Mumbai     450        3047          0.148           0.048
    1006 Mumbai    1298        3047          0.426           0.139
    1007  Delhi    2499        3298          0.758           0.268
    1008   Pune    1400        2995          0.467           0.150
{'Delhi': 3298.0, 'Mumbai': 1523.5, 'Pune': 1497.5}


In [ ]:
import pandas as pd
sales = pd.DataFrame({"City": ["Delhi", "Mumbai", "Pune"], "Sales": [3298, 3047, 2995]})
targets = pd.DataFrame({"City": ["Delhi", "Mumbai", "Mumbai", "Pune"],     # Mumbai has two target rows
                        "Target": [3000, 1500, 1500, 3200]})
bad = sales.merge(targets, on="City", how="left")
print(bad.to_string(index=False))
print("sales after blend:", bad.Sales.sum(), "(true 9340)")
good = sales.merge(targets.groupby("City", as_index=False).Target.sum(), on="City", how="left")
print(good.to_string(index=False))
print("sales after fix  :", good.Sales.sum())

  City  Sales  Target
 Delhi   3298    3000
Mumbai   3047    1500
Mumbai   3047    1500
  Pune   2995    3200
sales after blend: 12387 (true 9340)
  City  Sales  Target
 Delhi   3298    3000
Mumbai   3047    3000
  Pune   2995    3200
sales after fix  : 9340


In [ ]:
import pandas as pd
weights = {"Governed model + RLS": 0.35, "Microsoft 365 fit": 0.25, "Cost": 0.20, "Visual flexibility": 0.10, "Team skills": 0.10}
scores = pd.DataFrame({"Power BI":      [5, 5, 4, 3, 4],
                       "Tableau":       [4, 2, 2, 5, 2],
                       "Looker Studio": [2, 1, 5, 3, 3]}, index=list(weights))
w = pd.Series(weights)
print("weights add to", round(w.sum(), 2))
print((scores.mul(w, axis=0)).sum().round(2).sort_values(ascending=False).to_string())

weights add to 1.0
Power BI         4.50
Tableau          3.00
Looker Studio    2.55


## 11.13 How to Solve Any Report Design, Interactivity, Security, Power BI Service Problem

Every ticket you will ever get on this subject is one of four symptoms: *it looks wrong*, *it behaves wrong*, *the wrong person sees it* or *it arrives wrong* (late, stale, slow). The figure is your first decision step. The rest of this section gives the method, then eight recurring patterns with a solved example each.

In [ ]:
import pandas as pd
orders = pd.DataFrame({
    "OrderID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    "Date": pd.to_datetime(["2025-01-03", "2025-01-03", "2025-01-04", "2025-01-05",
                            "2025-01-06", "2025-01-07", "2025-01-09", "2025-01-10"]),
    "City": ["Pune", "Mumbai", "Pune", "Delhi", "Mumbai", "Mumbai", "Delhi", "Pune"],
    "Status": ["Delivered", "Delivered", "Returned", "Delivered", "Cancelled", "Delivered", "Delivered", "Delivered"],
    "Qty": [2, 1, 3, 1, 5, 2, 1, 4],
    "Amount": [998, 1299, 597, 799, 450, 1298, 2499, 1400]})
region = {"Delhi": "North", "Mumbai": "West", "Pune": "West"}
orders["Region"] = orders["City"].map(region)

target = 8293
hits = []
for col in ["City", "Status", "Region"]:
    for v in orders[col].unique():
        s = orders.loc[orders[col] == v, "Amount"].sum()
        if s == target:
            hits.append((col, v, int(s)))
print("single filter hits:", hits)
for col in ["City", "Status", "Region"]:
    for v in orders[col].unique():
        s = orders.loc[orders[col] != v, "Amount"].sum()
        if s == target:
            print("exclude", col, v, "->", s)

single filter hits: [('Status', 'Delivered', 8293)]


In [ ]:
from datetime import datetime, timedelta, timezone
IST = timezone(timedelta(hours=5, minutes=30))
UTC = timezone.utc
source_done_utc = datetime(2025, 1, 10, 0, 30, tzinfo=UTC)     # warehouse job finishes 00:30 UTC
start_ist = source_done_utc.astimezone(IST) + timedelta(minutes=15)
end_ist = start_ist + timedelta(minutes=25)
deadline = datetime(2025, 1, 10, 7, 0, tzinfo=IST)
print("source done (IST):", source_done_utc.astimezone(IST).strftime("%H:%M"))
print("refresh start    :", start_ist.strftime("%H:%M"), " end:", end_ist.strftime("%H:%M"))
print("slack (min)      :", int((deadline - end_ist).total_seconds() // 60))
print("if set 06:15 UTC read as IST by mistake, start:", datetime(2025, 1, 10, 6, 15, tzinfo=UTC).astimezone(IST).strftime("%H:%M"))

source done (IST): 06:00
refresh start    : 06:15  end: 06:40
slack (min)      : 20
if set 06:15 UTC read as IST by mistake, start: 11:45


In [ ]:
import pandas as pd
perf = pd.DataFrame({"visual": ["matrix", "map", "slicer", "line", "card"],
                     "dax_s": [7.8, 0.9, 0.2, 0.3, 0.1],
                     "display_s": [1.7, 1.2, 0.1, 0.4, 0.0]})
perf["total_s"] = perf.dax_s + perf.display_s
perf["share"] = (perf.total_s / perf.total_s.sum()).round(3)
perf["main_cost"] = perf[["dax_s", "display_s"]].idxmax(axis=1)
print(perf.sort_values("total_s", ascending=False).to_string(index=False))

visual  dax_s  display_s  total_s  share main_cost
matrix    7.8        1.7      9.5  0.748     dax_s
   map    0.9        1.2      2.1  0.165 display_s
  line    0.3        0.4      0.7  0.055 display_s
slicer    0.2        0.1      0.3  0.024     dax_s
  card    0.1        0.0      0.1  0.008     dax_s


In [ ]:
import pandas as pd
orders = pd.DataFrame({
    "OrderID": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    "Date": pd.to_datetime(["2025-01-03", "2025-01-03", "2025-01-04", "2025-01-05",
                            "2025-01-06", "2025-01-07", "2025-01-09", "2025-01-10"]),
    "City": ["Pune", "Mumbai", "Pune", "Delhi", "Mumbai", "Mumbai", "Delhi", "Pune"],
    "Status": ["Delivered", "Delivered", "Returned", "Delivered", "Cancelled", "Delivered", "Delivered", "Delivered"],
    "Qty": [2, 1, 3, 1, 5, 2, 1, 4],
    "Amount": [998, 1299, 597, 799, 450, 1298, 2499, 1400]})
region = {"Delhi": "North", "Mumbai": "West", "Pune": "West"}
orders["Region"] = orders["City"].map(region)

kept = orders[orders.Status != "Cancelled"]
aov = kept.groupby("City").Amount.sum() / kept.groupby("City").OrderID.nunique()
print(aov.round(2).to_dict())

{'Delhi': 1649.0, 'Mumbai': 1298.5, 'Pune': 998.33}


## 11.14 Going Deeper: Engine Internals, Edge Cases and What Interviewers Probe

In [ ]:
import math
import numpy as np, pandas as pd
rows = 1_000_000
for col, d in [("City", 3), ("Status", 3), ("Date", 365), ("ProductKey", 200), ("OrderID", rows), ("Amount (cents)", 900_000)]:
    bits = math.ceil(math.log2(d))
    print(f"{col:15s} distinct={d:>9,}  bits/row={bits:2d}  approx MB={rows * bits / 8 / 1e6:6.2f}")

rng = np.random.default_rng(0)
city = pd.Series(rng.choice(["Delhi", "Mumbai", "Pune"], rows))
print("object column MB  :", round(city.memory_usage(deep=True) / 1e6, 1))
print("category column MB:", round(city.astype("category").memory_usage(deep=True) / 1e6, 1))

City            distinct=        3  bits/row= 2  approx MB=  0.25
Status          distinct=        3  bits/row= 2  approx MB=  0.25
Date            distinct=      365  bits/row= 9  approx MB=  1.12
ProductKey      distinct=      200  bits/row= 8  approx MB=  1.00
OrderID         distinct=1,000,000  bits/row=20  approx MB=  2.50
Amount (cents)  distinct=  900,000  bits/row=20  approx MB=  2.50
object column MB  : 54.0
category column MB: 1.0


## 11.15 Mixed Practice Drill

Ten quick questions mixing all the sections. Try each on paper using the reference numbers (total 9,340; West 6,042; North 3,298; Delivered 8,293), then read the answer.

## 11.16 Self-Test and Scoring Guide

Close the chapter and answer on paper. One point per correct answer. Do not look at the key until you finish all 15.

## 11.17 Where This Chapter Fits: Before and After

## Chapter recap

## Practice set

Problem 11.1 Four cards on a 1280-pixel page with margins of 20 and gaps of 20. How wide is each card?

In [ ]:
def grid(total, margin, gap, n):
    width = (total - 2 * margin - (n - 1) * gap) / n
    return width, [margin + k * (width + gap) for k in range(n)]

w, starts = grid(1920, 24, 24, 4)
print(w, starts)

450.0 [24.0, 498.0, 972.0, 1446.0]


In [ ]:
import pandas as pd
sec = pd.DataFrame({"Email": ["cfo@corp.example", "cfo@corp.example", "asha@corp.example"],
                    "Region": ["West", "North ", "West"]})          # note the space after North
known = {"North", "West"}
for e, r in zip(sec.Email, sec.Region):
    print(repr(r), "matches a real region" if r in known else "NO MATCH")
print("cleaned:", sec.Region.str.strip().tolist())

'West' matches a real region
'North ' NO MATCH
'West' matches a real region
cleaned: ['West', 'North', 'West']


## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/analyst/ch11.html